<a href="https://colab.research.google.com/github/Viera1624/TelecomX_LATAM/blob/main/telecomx_latam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Trello del desafío](https://trello.com/b/Va6JwjjL/telecomxlatam)

#📌 Extracción

In [108]:
import pandas as pd

In [109]:
datos_clientes = pd.read_json('/content/TelecomX_Data.json')
datos_clientes.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


##Normalizando Json

In [110]:
def normalizar_json(columna):
  df_norm = pd.json_normalize(datos_clientes[columna])

  return df_norm

customer = normalizar_json('customer')
phone = normalizar_json('phone')
internet = normalizar_json('internet')
account = normalizar_json('account')


In [111]:
datos_clientes_norm = pd.concat([datos_clientes[['customerID', 'Churn']],
                                 customer, phone, internet, account],
                                axis=1)
datos_clientes_norm

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.60,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.90,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.90,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.00,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.90,267.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7262,9987-LUTYD,No,Female,0,No,No,13,Yes,No,DSL,...,No,No,Yes,No,No,One year,No,Mailed check,55.15,742.9
7263,9992-RRAMN,Yes,Male,0,Yes,No,22,Yes,Yes,Fiber optic,...,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,85.10,1873.7
7264,9992-UJOEL,No,Male,0,No,No,2,Yes,No,DSL,...,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,50.30,92.75
7265,9993-LHIEB,No,Male,0,Yes,Yes,67,Yes,No,DSL,...,No,Yes,Yes,No,Yes,Two year,No,Mailed check,67.85,4627.65


#🔧 Transformación

##Verificando si hay valores nulos, duplicados o errores de formato

In [112]:
datos_clientes_norm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7267 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contract          7267 non-null   object 


In [113]:
datos_clientes_norm.dtypes

,0
customerID,object
Churn,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object


Los IDs de cliente son valores únicos, por lo que no debería haber un ID presente más de una vez en nuestros registros

In [114]:
x =datos_clientes_norm['customerID'].unique()

In [115]:
len(x)

7267

##Normalizando las columnaas

###Renombrar las columnaas

In [116]:
columnas = list(datos_clientes_norm.columns)

columnas_nuevas = ['ID_Cliente','Retirado','Genero','Mayor_65_Años','Tiene_Pareja','Dependientes',
                    'Meses_De_Contrato','Servicio_Telefonico','Multiples_Lineas','Servicio_Internet','Seguridad_En_linea',
                    'Respaldo_En_linea','Proteccion_Del_Dispositivo','Soporte_Tecnico','Television_Por_Cable','Streaming_Peliculas',
                    'Tipo_De_Contrato', 'Factura_En_Linea','Metodo_De_Pago', 'Total_Por_Mes',
                    'Total_Gastado']

renombrar = dict(zip(columnas,columnas_nuevas))

In [117]:
datos_clientes_norm.rename(columns=renombrar, inplace=True)
datos_clientes_norm.head(1)

,ID_Cliente,Retirado,Genero,Mayor_65_Años,Tiene_Pareja,Dependientes,Meses_De_Contrato,Servicio_Telefonico,Multiples_Lineas,Servicio_Internet,...,Respaldo_En_linea,Proteccion_Del_Dispositivo,Soporte_Tecnico,Television_Por_Cable,Streaming_Peliculas,Tipo_De_Contrato,Factura_En_Linea,Metodo_De_Pago,Total_Por_Mes,Total_Gastado
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3


##Transformando datos numéricos

Alintentar transformar el tipo de datos de la columna Total_Gastado a float encontramos que hay espacios entre los registros, vamos a verificar

In [118]:
datos_clientes_norm[datos_clientes_norm['Total_Gastado'].str.contains('\ ', na= False)]

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-4215343052.py:1: SyntaxWarning: invalid escape sequence '\ '
  datos_clientes_norm[datos_clientes_norm['Total_Gastado'].str.contains('\ ', na= False)]


,ID_Cliente,Retirado,Genero,Mayor_65_Años,Tiene_Pareja,Dependientes,Meses_De_Contrato,Servicio_Telefonico,Multiples_Lineas,Servicio_Internet,...,Respaldo_En_linea,Proteccion_Del_Dispositivo,Soporte_Tecnico,Television_Por_Cable,Streaming_Peliculas,Tipo_De_Contrato,Factura_En_Linea,Metodo_De_Pago,Total_Por_Mes,Total_Gastado
975,1371-DWPAZ,No,Female,0,Yes,Yes,0,No,No phone service,DSL,...,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,
1775,2520-SGTTA,No,Female,0,Yes,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,
1955,2775-SEFEE,No,Male,0,No,Yes,0,Yes,Yes,DSL,...,Yes,No,Yes,No,No,Two year,Yes,Bank transfer (automatic),61.90,
2075,2923-ARZLG,No,Male,0,Yes,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,
2232,3115-CZMZD,No,Male,0,No,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,
2308,3213-VVOLG,No,Male,0,Yes,Yes,0,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,
2930,4075-WKNIU,No,Female,0,Yes,Yes,0,Yes,Yes,DSL,...,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,
3134,4367-NUYAO,No,Male,0,Yes,Yes,0,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,
3203,4472-LVYGI,No,Female,0,Yes,Yes,0,No,No phone service,DSL,...,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,
4169,5709-LVOEQ,No,Female,0,Yes,Yes,0,Yes,No,DSL,...,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,


Una vez encontrados los registros que tienen espacios, podemos observar que coinciden con el numero 0(cero) en la columna Meses_De_Contrato, esto puede indicar que son clientes nuevos y por lo tanto, no han acumulado un cargo total. Vamos a verificar si hay más clientes nuevos

In [119]:
datos_clientes_norm.query('Meses_De_Contrato == 0')

,ID_Cliente,Retirado,Genero,Mayor_65_Años,Tiene_Pareja,Dependientes,Meses_De_Contrato,Servicio_Telefonico,Multiples_Lineas,Servicio_Internet,...,Respaldo_En_linea,Proteccion_Del_Dispositivo,Soporte_Tecnico,Television_Por_Cable,Streaming_Peliculas,Tipo_De_Contrato,Factura_En_Linea,Metodo_De_Pago,Total_Por_Mes,Total_Gastado
975,1371-DWPAZ,No,Female,0,Yes,Yes,0,No,No phone service,DSL,...,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,
1775,2520-SGTTA,No,Female,0,Yes,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,
1955,2775-SEFEE,No,Male,0,No,Yes,0,Yes,Yes,DSL,...,Yes,No,Yes,No,No,Two year,Yes,Bank transfer (automatic),61.90,
2075,2923-ARZLG,No,Male,0,Yes,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,
2232,3115-CZMZD,No,Male,0,No,Yes,0,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,
2308,3213-VVOLG,No,Male,0,Yes,Yes,0,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,
2930,4075-WKNIU,No,Female,0,Yes,Yes,0,Yes,Yes,DSL,...,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,
3134,4367-NUYAO,No,Male,0,Yes,Yes,0,Yes,Yes,No,...,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,
3203,4472-LVYGI,No,Female,0,Yes,Yes,0,No,No phone service,DSL,...,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,
4169,5709-LVOEQ,No,Female,0,Yes,Yes,0,Yes,No,DSL,...,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,


los resultados obtenidos coinciden con ceros en la columna Meses_De_Contrato y espacios en Total_Gastado, procedemos entonces a reemplazar espacios por ceros y realizar la respectiva transformación

In [120]:
datos_clientes_norm['Total_Gastado']= datos_clientes_norm['Total_Gastado'].replace('\ ', '0', regex=True)


<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-3318985259.py:1: SyntaxWarning: invalid escape sequence '\ '
  datos_clientes_norm['Total_Gastado']= datos_clientes_norm['Total_Gastado'].replace('\ ', '0', regex=True)


In [121]:
import numpy as np
datos_clientes_norm['Total_Gastado']= datos_clientes_norm['Total_Gastado'].astype(np.float64)

Ahora, tenemos una serie de columnas con valores de "Yes" y "No", pero para este análisis nos resultaría más fácil trabajar con 0 para "No" y 1 para "Yes", así que vamos a reemplazar estos valores creando una lista con dichas columnas que queremos transformar de str a int64

In [122]:
transformar = ['Retirado','Tiene_Pareja','Dependientes',
               'Servicio_Telefonico','Multiples_Lineas','Seguridad_En_linea',
               'Respaldo_En_linea','Proteccion_Del_Dispositivo','Soporte_Tecnico','Television_Por_Cable','Streaming_Peliculas',
               'Factura_En_Linea']

Verificamos los valores únicos en las columnasde nuestra lista para identificar las posibles complicaciones (Valores distintos a "Yes" o "No") al momento de realizar la transformación

In [123]:
for columna in datos_clientes_norm[transformar]:
    print(f"Valores unicos para la columna '{columna}':")
    print(datos_clientes_norm[columna].unique())
    print("\n")

Valores unicos para la columna 'Retirado':
['No' 'Yes' '']


Valores unicos para la columna 'Tiene_Pareja':
['Yes' 'No']


Valores unicos para la columna 'Dependientes':
['Yes' 'No']


Valores unicos para la columna 'Servicio_Telefonico':
['Yes' 'No']


Valores unicos para la columna 'Multiples_Lineas':
['No' 'Yes' 'No phone service']


Valores unicos para la columna 'Seguridad_En_linea':
['No' 'Yes' 'No internet service']


Valores unicos para la columna 'Respaldo_En_linea':
['Yes' 'No' 'No internet service']


Valores unicos para la columna 'Proteccion_Del_Dispositivo':
['No' 'Yes' 'No internet service']


Valores unicos para la columna 'Soporte_Tecnico':
['Yes' 'No' 'No internet service']


Valores unicos para la columna 'Television_Por_Cable':
['Yes' 'No' 'No internet service']


Valores unicos para la columna 'Streaming_Peliculas':
['No' 'Yes' 'No internet service']


Valores unicos para la columna 'Factura_En_Linea':
['Yes' 'No']




Como podemos observar, hay columnas que tienen cadenas vacías (''), entre otras como 'No internet service' y 'No phone service', tenemos que reemplazar también estos valores por 0 para poder realizar nuestra transformación

In [124]:
#Creamos un diccionario con los valores a reemplazar
reemplazar = {'': '0', 'No phone service': '0', 'No internet service': '0', 'Yes': '1', 'No': '0'}

#Reemplazamos los valores con un replace() dentro de un lazo for
for columna in datos_clientes_norm[transformar]:
  datos_clientes_norm[transformar] = datos_clientes_norm[transformar].replace(reemplazar)


In [125]:
for columna in datos_clientes_norm[transformar]:
    print(f"Valores unicos para la columna '{columna}':")
    print(datos_clientes_norm[columna].unique())
    print("\n")

Valores unicos para la columna 'Retirado':
['0' '1']


Valores unicos para la columna 'Tiene_Pareja':
['1' '0']


Valores unicos para la columna 'Dependientes':
['1' '0']


Valores unicos para la columna 'Servicio_Telefonico':
['1' '0']


Valores unicos para la columna 'Multiples_Lineas':
['0' '1']


Valores unicos para la columna 'Seguridad_En_linea':
['0' '1']


Valores unicos para la columna 'Respaldo_En_linea':
['1' '0']


Valores unicos para la columna 'Proteccion_Del_Dispositivo':
['0' '1']


Valores unicos para la columna 'Soporte_Tecnico':
['1' '0']


Valores unicos para la columna 'Television_Por_Cable':
['1' '0']


Valores unicos para la columna 'Streaming_Peliculas':
['0' '1']


Valores unicos para la columna 'Factura_En_Linea':
['1' '0']




In [128]:
datos_clientes_norm[transformar] = datos_clientes_norm[transformar].astype(np.int64)

In [129]:
datos_clientes_norm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID_Cliente                  7267 non-null   object 
 1   Retirado                    7267 non-null   int64  
 2   Genero                      7267 non-null   object 
 3   Mayor_65_Años               7267 non-null   int64  
 4   Tiene_Pareja                7267 non-null   int64  
 5   Dependientes                7267 non-null   int64  
 6   Meses_De_Contrato           7267 non-null   int64  
 7   Servicio_Telefonico         7267 non-null   int64  
 8   Multiples_Lineas            7267 non-null   int64  
 9   Servicio_Internet           7267 non-null   object 
 10  Seguridad_En_linea          7267 non-null   int64  
 11  Respaldo_En_linea           7267 non-null   int64  
 12  Proteccion_Del_Dispositivo  7267 non-null   int64  
 13  Soporte_Tecnico             7267 

##Creacion de columna Cuentas_Diarias

In [106]:
datos_clientes_norm['Cuentas_Diarias'] = (datos_clientes_norm['Total_Por_Mes']/30).round(2)


In [107]:
datos_clientes_norm.head(2)

,ID_Cliente,Retirado,Genero,Mayor_65_Años,Tiene_Pareja,Dependientes,Meses_De_Contrato,Servicio_Telefonico,Multiples_Lineas,Servicio_Internet,...,Proteccion_Del_Dispositivo,Soporte_Tecnico,Television_Por_Cable,Streaming_Peliculas,Tipo_De_Contrato,Factura_En_Linea,Metodo_De_Pago,Total_Por_Mes,Total_Gastado,Cuentas_Diarias
0,0002-ORFBO,0,Female,0,1,1,9,1,0,DSL,...,0,1,1,0,One year,1,Mailed check,65.6,593.3,2.19
1,0003-MKNFE,0,Male,0,0,0,9,1,1,DSL,...,0,0,0,1,Month-to-month,0,Mailed check,59.9,542.4,2.00


#📊 Carga y análisis

#📄Informe final